# Prepare a reproducible LLM pilot sample

This notebook continues the completed inspection and full chess validation. It prepares **75 puzzles: 25 per rating stratum**, using seed 401. It makes no LLM calls and incurs no inference charges.

**Provisional eligibility:** all validated puzzles. No rating-quality threshold is imposed before researcher review. Rating deviation and play counts are summarized below to support that decision. If thresholds change, rerun sampling before collecting outcomes. A balanced sample describes these strata, not the natural puzzle frequency distribution.

Run with the project `.venv` kernel, from the project root or `code/`.

In [1]:
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import hashlib
import json
import math
import numpy as np
import pandas as pd
import chess
from IPython.display import display

root = Path.cwd()
if root.name == 'code':
    root = root.parent
feature_path = root / 'data/validation/full/validated_features.csv'
source_path = root / 'data/lichess_db_puzzle.csv.zst'
summary = json.loads((feature_path.parent / 'summary.json').read_text())
assert summary['scope'] == 'full_dataset' and summary['failed'] == 0
SEED = 401
PER_STRATUM = 25
CHUNK_SIZE = 100_000
# Optional eligibility settings; None preserves all validated records.
MAX_RATING_DEVIATION = None
MIN_PLAYS = None
out = root / 'data/samples/pilot_v1'
out.mkdir(parents=True, exist_ok=True)

def eligible(frame):
    mask = pd.Series(True, index=frame.index)
    if MAX_RATING_DEVIATION is not None:
        mask &= frame.rating_deviation <= MAX_RATING_DEVIATION
    if MIN_PLAYS is not None:
        mask &= frame.nb_plays >= MIN_PLAYS
    return frame.loc[mask].copy()
print('Source:', feature_path)
print('Eligibility: provisional, no filters' if MAX_RATING_DEVIATION is None and MIN_PLAYS is None else 'Eligibility filters enabled')

Source: E:\DA_401\LLM_Chess_Comparison\data\validation\full\validated_features.csv
Eligibility: provisional, no filters


## 1. Inspect the eligible population and define rating strata

Scan in chunks to avoid loading the full feature table into memory. Rating cutoffs are the nearest-rank one-third and two-thirds quantiles of the eligible population. Tied ratings remain together; population stratum counts need not be equal.

In [2]:
counts = {key: Counter() for key in ['puzzle_rating', 'rating_deviation', 'nb_plays']}
processed = population_n = 0
for chunk in pd.read_csv(feature_path, chunksize=CHUNK_SIZE,
                         usecols=list(counts), dtype={key: 'int64' for key in counts}):
    processed += len(chunk)
    frame = eligible(chunk)
    population_n += len(frame)
    for key in counts:
        counts[key].update(frame[key].value_counts().to_dict())
assert processed == summary['valid']
assert population_n >= PER_STRATUM * 3

def quantile(counter, q):
    target = max(1, math.ceil(sum(counter.values()) * q))
    cumulative = 0
    for value, n in sorted(counter.items()):
        cumulative += n
        if cumulative >= target:
            return int(value)

lower = quantile(counts['puzzle_rating'], 1/3)
upper = quantile(counts['puzzle_rating'], 2/3)
assert lower < upper
def strata(ratings):
    return pd.Series(np.where(ratings <= lower, 'low',
                     np.where(ratings <= upper, 'middle', 'high')), index=ratings.index)
population_counts = {'low': 0, 'middle': 0, 'high': 0}
for rating, n in counts['puzzle_rating'].items():
    population_counts['low' if rating <= lower else 'middle' if rating <= upper else 'high'] += n
display(pd.DataFrame({key: {str(q): quantile(counter, q) for q in [.05, .25, .5, .75, .95]}
                      for key, counter in counts.items()}).rename_axis('quantile'))
print(f'Validated: {processed:,}; eligible: {population_n:,}; excluded: {processed-population_n:,}')
print(f'Low <= {lower}; middle > {lower} and <= {upper}; high > {upper}')
display(pd.Series(population_counts, name='eligible_puzzles').to_frame())

,puzzle_rating,rating_deviation,nb_plays
quantile,,,
0.05,648,75,25
0.25,1041,76,121
0.5,1413,79,400
0.75,1870,89,1336
0.95,2437,118,8440


Validated: 6,157,341; eligible: 6,157,341; excluded: 0
Low <= 1157; middle > 1157 and <= 1700; high > 1700


,eligible_puzzles
low,2056407
middle,2050659
high,2050275


## 2. Sample across the full file and derive game phase

Assign independent random priorities to all eligible rows and retain the 25 smallest in each stratum. This samples uniformly within strata without taking the first rows. Reproducibility depends on seed, source row order, and the saved software versions.

Game phase uses exact theme tokens. Missing phase tags become `unknown`; multiple phase tags become `mixed`. Themes are multi-label and are not forced into a single tactical category.

In [3]:
rng = np.random.default_rng(SEED)
selected = {name: pd.DataFrame() for name in population_counts}
for chunk in pd.read_csv(feature_path, chunksize=CHUNK_SIZE, dtype={'puzzle_id': str}):
    frame = eligible(chunk)
    frame['rating_stratum'] = strata(frame.puzzle_rating)
    frame['_priority'] = rng.random(len(frame))
    for name in selected:
        candidates = frame.loc[frame.rating_stratum.eq(name)].nsmallest(PER_STRATUM, '_priority')
        selected[name] = pd.concat([selected[name], candidates], ignore_index=True).nsmallest(PER_STRATUM, '_priority')
pilot = pd.concat([selected[name] for name in ['low', 'middle', 'high']], ignore_index=True)
pilot = pilot.drop(columns='_priority').sort_values(['rating_stratum', 'puzzle_id']).reset_index(drop=True)
def phase(text):
    tags = set(str(text).split()) & {'opening', 'middlegame', 'endgame'}
    return next(iter(tags)) if len(tags) == 1 else 'unknown' if not tags else 'mixed'
pilot['game_phase'] = pilot.themes.map(phase)
assert len(pilot) == 3 * PER_STRATUM and pilot.puzzle_id.is_unique
assert pilot.groupby('rating_stratum').size().eq(PER_STRATUM).all()
display(pilot.groupby('rating_stratum').agg(n=('puzzle_id','size'),
    min_rating=('puzzle_rating','min'), max_rating=('puzzle_rating','max'),
    median_deviation=('rating_deviation','median'), min_plays=('nb_plays','min')))
display(pd.crosstab(pilot.rating_stratum, pilot.game_phase))
display(pd.crosstab(pilot.rating_stratum, pilot.total_solver_moves))
display(pd.Series(Counter(tag for text in pilot.themes for tag in text.split()),
                  name='puzzle_count').sort_values(ascending=False).head(20).to_frame())

,n,min_rating,max_rating,median_deviation,min_plays
rating_stratum,,,,,
high,25,1713,2671,78.0,26
low,25,446,1146,79.0,35
middle,25,1178,1682,76.0,20


game_phase,endgame,middlegame,opening
rating_stratum,,,
high,14,9,2
low,10,13,2
middle,14,9,2


total_solver_moves,1,2,3,4,5
rating_stratum,,,,,
high,0,12,9,3,1
low,10,14,1,0,0
middle,0,15,9,1,0


,puzzle_count
short,41
endgame,38
middlegame,31
crushing,28
mate,24
advantage,23
long,19
fork,12
mateIn1,10
sacrifice,10


## 3. Recover official answers and verify every sampled puzzle

The validated feature file does not contain official moves. Recover original FEN, Moves, and GameUrl by exact puzzle ID from the compressed source. Replay all moves and verify that the reconstructed solver position matches the saved feature.

The output below is a **researcher answer key**. Never include official moves, ratings, themes, IDs, or URLs in the model prompt.

In [4]:
wanted = set(pilot.puzzle_id)
matches = []
for chunk in pd.read_csv(source_path, compression='zstd', chunksize=CHUNK_SIZE,
                         usecols=['PuzzleId','FEN','Moves','GameUrl'], dtype=str, keep_default_na=False):
    found = chunk.loc[chunk.PuzzleId.isin(wanted)]
    if not found.empty:
        matches.append(found)
answers = pd.concat(matches, ignore_index=True).rename(columns={
    'PuzzleId':'puzzle_id', 'FEN':'original_fen', 'Moves':'official_moves', 'GameUrl':'game_url'})
assert len(answers) == len(wanted) and answers.puzzle_id.is_unique
pilot = pilot.merge(answers, on='puzzle_id', validate='one_to_one')
for row in pilot.itertuples(index=False):
    board = chess.Board(row.original_fen)
    moves = row.official_moves.split()
    assert board.is_valid() and len(moves) >= 2
    for i, text in enumerate(moves):
        move = chess.Move.from_uci(text)
        assert move in board.legal_moves, (row.puzzle_id, i, text)
        board.push(move)
        if i == 0:
            assert board.fen() == row.solver_fen
            assert ('white' if board.turn else 'black') == row.solver_side
            assert board.legal_moves.count() == row.legal_moves
            assert len(board.piece_map()) == row.piece_count
    assert len(moves[1::2]) == row.total_solver_moves
print(f'All {len(pilot)} sampled puzzles joined and replayed successfully.')

All 75 sampled puzzles joined and replayed successfully.


## 4. Save the sample, safe model inputs, and provenance

Rerunning with identical inputs overwrites these versioned preparation outputs. Freeze or copy this folder before collecting outcomes. The model-input file intentionally contains only the starting position and side to move; use its ID for internal bookkeeping, not prompt content.

In [5]:
checksum = hashlib.sha256()
with source_path.open('rb') as stream:
    for block in iter(lambda: stream.read(1024 * 1024), b''):
        checksum.update(block)
assert checksum.hexdigest() == summary['sha256'], 'Source snapshot changed; revalidate before sampling.'
pilot.to_csv(out / 'pilot_puzzles.csv', index=False)
pilot[['puzzle_id','solver_fen','solver_side']].to_csv(out / 'pilot_model_inputs.csv', index=False)
metadata = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'stage': 'prepared_for_researcher_review_no_llm_calls',
    'seed': SEED, 'per_stratum': PER_STRATUM, 'sample_size': len(pilot),
    'source_sha256': checksum.hexdigest(), 'source': str(source_path),
    'validated_rows': processed, 'eligible_rows': population_n,
    'max_rating_deviation': MAX_RATING_DEVIATION, 'min_plays': MIN_PLAYS,
    'rating_cutoffs': {'lower_inclusive': lower, 'upper_inclusive': upper},
    'population_stratum_counts': population_counts,
    'sampling_method': 'smallest independent uniform random priorities within rating strata',
    'game_phase_rule': 'exact phase theme token; unknown if none; mixed if multiple',
    'versions': {'pandas': pd.__version__, 'numpy': np.__version__, 'chess': chess.__version__},
    'notes': ['Eligibility is provisional pending researcher review.',
              'Equal allocation across strata does not represent natural puzzle frequencies.',
              'No LLM responses, success labels, or fitted models exist at this stage.']
}
(out / 'pilot_sampling_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
saved = pd.read_csv(out / 'pilot_puzzles.csv', dtype={'puzzle_id':str})
assert len(saved) == len(pilot) and set(saved.puzzle_id) == wanted
assert saved.notna().all().all()
print('Saved and checked:', out)
display(saved[['puzzle_id','rating_stratum','puzzle_rating','total_solver_moves','game_phase']].head(10))

Saved and checked: E:\DA_401\LLM_Chess_Comparison\data\samples\pilot_v1


,puzzle_id,rating_stratum,puzzle_rating,total_solver_moves,game_phase
0,1Z6xa,high,2195,4,endgame
1,1diZo,high,1899,2,endgame
2,1rpa1,high,1955,3,middlegame
3,20zH3,high,2209,5,endgame
4,4ShCG,high,2430,4,endgame
5,7Geqj,high,2192,3,endgame
6,9gbYy,high,2671,2,endgame
7,AaUSH,high,2181,2,opening
8,D99Tg,high,1759,2,endgame
9,HZbLU,high,1959,2,endgame


## Review checkpoint — ready for approval

Completed: population quality summaries, rating strata, reproducible 75-puzzle sample, game-phase derivation, official-answer recovery, replay checks, and saved inputs/provenance.

Before LLM collection, review eligibility and sample coverage. Next approve the actual LLM endpoints/checkpoints, shared prompt/board representation, generation settings and budget, and official-line scoring rules. Then build the resumable runner, manually inspect 5–10 trials, and run this pilot. Infrastructure errors must remain separate from chess failures.

After valid outcomes exist: join one row per puzzle–LLM pair, summarize success rates, split by puzzle ID, fit logistic regression with puzzle-clustered standard errors, and fit a shallow decision tree followed by the planned XGBoost model. No predictive results can be computed until actual success/failure outcomes are collected.